# 01 · Ingest games and quarter scores (ESPN)

Every FBS game since 2015: schedule, venue, neutral site, conference game, final score and
**points by quarter** (Q1–Q4 + OT). Source: ESPN scoreboard (free, no key).

- Output: `data/raw/games.parquet`, one row per game, home/away columns.
- Raw JSON is cached in `data/raw/espn/`. Finished weeks are never downloaded again, so
  re-running this notebook each week only pulls the new games.
- Betting lines are **not** here. They come from CFBD in `02_ingest_lines`.

In [ ]:
import pandas as pd

from canes_cfb.espn import load_seasons
from canes_cfb.paths import RAW

SEASONS = list(range(2015, 2027))

## 1. Download
First run takes ~1.5 min (~200 requests). After that it's mostly cache.

In [ ]:
games = load_seasons(SEASONS)
print(f"{len(games):,} games | {games.completed.sum():,} completed")
games.tail()

## 2. Validate

These checks must pass before anything downstream uses the data:
1. `game_id` is unique.
2. Q1 + Q2 + Q3 + Q4 + OT = final score for every completed, full-length game.
3. FBS team counts per season look right (~128 in 2015, growing as schools move up).
4. Games shortened by weather (`shortened`) are flagged. Quarter targets don't exist for them.

In [ ]:
assert games.game_id.is_unique

done = games[games.completed & ~games.shortened]
for side in ("home", "away"):
    by_period = done[[f"{side}_q{i}" for i in range(1, 5)]].sum(axis=1) + done[f"{side}_ot"]
    mismatches = (by_period != done[f"{side}_points"]).sum()
    assert mismatches == 0, f"{side}: {mismatches} games where quarters != final"

fbs_games = games[games.home_fbs & games.away_fbs]
fbs_teams = pd.concat(
    [
        fbs_games[["season", "home_id"]].set_axis(["season", "team_id"], axis=1),
        fbs_games[["season", "away_id"]].set_axis(["season", "team_id"], axis=1),
    ]
).drop_duplicates()
print("FBS teams per season:", fbs_teams.groupby("season").size().to_dict())
print("Shortened by weather:")
games[games.shortened][["season", "week", "home_team", "away_team", "home_points", "away_points"]]

## 3. Status of games that didn't finish
Canceled and postponed games (mostly 2020) stay in the table with `completed = False`.

In [ ]:
games[~games.completed].groupby(["season", "status"]).size().unstack(fill_value=0)

## 4. Save

In [ ]:
games.to_parquet(RAW / "games.parquet", index=False)
print("saved", RAW / "games.parquet", games.shape)